# Interactive Debug Viewer

This notebook reads `debug_upper_level_preupdate.json` and lets you click on a time point (upper-level update day) to inspect demand, capacities, waiting times, P&L, probabilities, and utility decomposition.

In [23]:
import json
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, Markdown

# Update this if you want to inspect another scenario folder
RESULT_DIR = './2-Results/tnc_capacity_1600'
DEBUG_FILE = os.path.join(RESULT_DIR, 'debug_upper_level_preupdate.json')

if not os.path.exists(DEBUG_FILE):
    raise FileNotFoundError(f'Debug file not found: {DEBUG_FILE}')

with open(DEBUG_FILE, 'r') as f:
    payload = json.load(f)

events = payload.get('events', [])
if not events:
    raise ValueError('No debug events found in JSON file.')

timeline_rows = []
for idx, ev in enumerate(events):
    totals = ev['allocation']['total_by_service']
    timeline_rows.append({
        'idx': idx,
        'update_day': ev['update_day'],
        'TNC': totals['TNC'],
        'MT': totals['MT'],
        'MaaS': totals['MaaS'],
    })

timeline_df = pd.DataFrame(timeline_rows).sort_values('update_day').reset_index(drop=True)
display(Markdown(f'Loaded **{len(timeline_df)}** upper-level debug events from `{DEBUG_FILE}`'))

Loaded **1155** upper-level debug events from `./2-Results/tnc_capacity_1600/debug_upper_level_preupdate.json`

In [ ]:
# Timeline figure (click a point to update all panels)
timeline_fig = go.FigureWidget()
for service in ['TNC', 'MT', 'MaaS']:
    timeline_fig.add_scatter(
        x=timeline_df['update_day'],
        y=timeline_df[service],
        mode='lines+markers',
        name=service,
        customdata=timeline_df['idx'],
    )

timeline_fig.update_layout(
    title='Demand Repartition Over Upper-Level Update Days (Click a marker)',
    xaxis_title='Upper-Level Update Day',
    yaxis_title='Demand Total',
    template='plotly_white',
    height=420,
)

detail_fig = go.FigureWidget(make_subplots(
    rows=2, cols=2,
    subplot_titles=('Demand Totals', 'Capacity (vkm)', 'Waiting Times (hr)', 'Operator P&L')
))
detail_fig.update_layout(template='plotly_white', height=650, showlegend=False)

prob_fig = go.FigureWidget()
prob_fig.update_layout(
    title='Choice Probabilities by Traveler Type',
    barmode='group',
    xaxis_title='Traveler Type',
    yaxis_title='Probability',
    template='plotly_white',
    height=380
)

utility_out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})
header = widgets.HTML('<b>Utility Decomposition (Selected Event)</b>')

def _utility_table_for_event(ev):
    rows = []
    for traveler in ev['utility_decomposition']:
        t = traveler['traveler_type'] + 1
        for svc, vals in traveler['services'].items():
            rows.append({
                'Type': t,
                'Service': svc,
                'ASC': vals['ASC'],
                'fare': vals['fare_disutility'],
                'time': vals['travel_time_disutility'],
                'wait': vals['waiting_time_disutility'],
                'U': vals['total_utility'],
            })
            if svc == 'MaaS' and 'mode_split' in vals:
                m = vals['mode_split']
                rows.append({
                    'Type': t,
                    'Service': 'MaaS split',
                    'ASC': None,
                    'fare': m['trip_fare'],
                    'time': m['travel_time_disutility_tnc'] + m['travel_time_disutility_mt'],
                    'wait': m['waiting_disutility_tnc'] + m['waiting_disutility_mt'],
                    'U': None,
                })
    df = pd.DataFrame(rows)
    return df

def update_dashboard(event_index):
    ev = events[event_index]

    # Demand and capacity
    demand_totals = ev['allocation']['total_by_service']
    cap_tnc = ev['capacity']['tnc']
    cap_maas = ev['capacity']['maas']
    wait = ev['waiting_times']
    fin = ev['financials']

    with detail_fig.batch_update():
        detail_fig.data = []

        detail_fig.add_bar(
            x=['TNC', 'MT', 'MaaS'],
            y=[demand_totals['TNC'], demand_totals['MT'], demand_totals['MaaS']],
            row=1, col=1,
            marker_color=['#1f77b4', '#2ca02c', '#ff7f0e']
        )

        detail_fig.add_bar(
            x=['TNC demand', 'TNC capacity', 'MaaS(TNC) demand', 'MaaS purchased'],
            y=[
                cap_tnc['demand_vkm'],
                cap_tnc['capacity_for_tnc_vkm'],
                cap_maas['demand_vkm'],
                cap_maas['purchased_tnc_capacity_vkm'],
            ],
            row=1, col=2,
            marker_color=['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']
        )

        detail_fig.add_bar(
            x=['TNC mean', 'MaaS mean'],
            y=[wait['tnc_mean_hr'], wait['maas_mean_hr']],
            row=2, col=1,
            marker_color=['#1f77b4', '#ff7f0e']
        )

        detail_fig.add_bar(
            x=['TNC rider_rev', 'TNC cap_sale', 'TNC op_cost', 'TNC net', 'MaaS rider_rev', 'MaaS tnc_cost', 'MaaS mt_cost', 'MaaS net'],
            y=[
                fin['tnc']['rider_revenue'],
                fin['tnc']['capacity_sale_revenue'],
                fin['tnc']['operating_cost'],
                fin['tnc']['net_profit'],
                fin['maas']['rider_revenue'],
                fin['maas']['tnc_capacity_purchase_cost'],
                fin['maas']['mt_capacity_purchase_cost'],
                fin['maas']['net_profit'],
            ],
            row=2, col=2,
            marker_color=['#2ca02c', '#2ca02c', '#d62728', '#9467bd', '#2ca02c', '#d62728', '#d62728', '#9467bd']
        )

        detail_fig.update_layout(title=f"Event at Update Day {ev['update_day']}")

    # Probabilities
    probs = ev['choice_probabilities']
    types = [f"Type {p['traveler_type']}" for p in probs]
    tnc_vals = [p['probabilities'][0] for p in probs]
    mt_vals = [p['probabilities'][1] for p in probs]
    maas_vals = [p['probabilities'][2] for p in probs]

    with prob_fig.batch_update():
        prob_fig.data = []
        prob_fig.add_bar(name='TNC', x=types, y=tnc_vals)
        prob_fig.add_bar(name='MT', x=types, y=mt_vals)
        prob_fig.add_bar(name='MaaS', x=types, y=maas_vals)
        prob_fig.update_layout(barmode='group')

    # Utility decomposition table
    util_df = _utility_table_for_event(ev)
    with utility_out:
        utility_out.clear_output(wait=True)
        display(util_df.style.format({
            'ASC': '{:.4f}',
            'fare': '{:.4f}',
            'time': '{:.4f}',
            'wait': '{:.4f}',
            'U': '{:.4f}',
        }))

def _on_click(trace, points, _selector):
    if not points.point_inds:
        return
    i = points.point_inds[0]
    idx = int(trace.customdata[i])
    update_dashboard(idx)

for tr in timeline_fig.data:
    tr.on_click(_on_click)

update_dashboard(0)
display(timeline_fig)
display(detail_fig)
display(prob_fig)
display(header)
display(utility_out)

FigureWidget({
    'data': [{'customdata': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'R4BHkEegR7BHwEfQR+BH8EgASBBIIE'),
                             'dtype': 'i2'},
              'mode': 'lines+markers',
              'name': 'TNC',
              'type': 'scatter',
              'uid': '5624b261-e1df-4b2e-86a5-6de6fc38b6af',
              'x': {'bdata': ('yAAAAIoBAABLAgAAEAMAANkDAACjBA' ... 'kAAI3pAAC/6QAA8ekAACPqAABV6gAA'),
                    'dtype': 'i4'},
              'y': {'bdata': ('yKE+O2RSIUCaeI03BD4kQNlMveEVEi' ... 'sPGbCMU0AJCM8XsIxTQCOzkhawjFNA'),
                    'dtype': 'f8'}},
             {'customdata': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'R4BHkEegR7BHwEfQR+BH8EgASBBIIE'),
                             'dtype': 'i2'},
              'mode': 'lines+markers',
              'name': 'MT',
              'type': 'scatter',
              'uid': 'f8802564-a44a-4dc1-8740-628e78ece4a5',
              'x': {'bdata': ('yAAAAIoBAABLAgAAEAMAANkDAACjBA' ... 'kAAI3

FigureWidget({
    'data': [{'marker': {'color': ['#1f77b4', '#2ca02c', '#ff7f0e']},
              'type': 'bar',
              'uid': 'ee94d069-aeea-485b-a116-6d20ff526d56',
              'x': [TNC, MT, MaaS],
              'xaxis': 'x',
              'y': [8.660920955081465, 34.6999109972961, 36.63916804762237],
              'yaxis': 'y'},
             {'marker': {'color': ['#1f77b4', '#aec7e8', '#ff7f0e', '#ffbb78']},
              'type': 'bar',
              'uid': 'a3de6c43-8f77-47d8-885c-5dd8e96b33a7',
              'x': [TNC demand, TNC capacity, MaaS(TNC) demand, MaaS purchased],
              'xaxis': 'x2',
              'y': [173.2184191016293, 960.0, 366.3916804762237, 640.0],
              'yaxis': 'y2'},
             {'marker': {'color': ['#1f77b4', '#ff7f0e']},
              'type': 'bar',
              'uid': '601288f3-ab9d-4e8e-9aa3-4ab00d5e1c4e',
              'x': [TNC mean, MaaS mean],
              'xaxis': 'x3',
              'y': [0.05041826605849172, 0.21049689

FigureWidget({
    'data': [{'name': 'TNC',
              'type': 'bar',
              'uid': '394ab527-b632-4de8-9d6d-8e95415d54f7',
              'x': [Type 1],
              'y': [0.11243954348734304]},
             {'name': 'MT',
              'type': 'bar',
              'uid': 'd0de98d1-b075-441c-a610-d6ef2cdc1987',
              'x': [Type 1],
              'y': [0.43284873767844373]},
             {'name': 'MaaS',
              'type': 'bar',
              'uid': 'f4665d6a-0abc-4ad0-a6d4-f085808e89a8',
              'x': [Type 1],
              'y': [0.45471171883421313]}],
    'layout': {'barmode': 'group',
               'height': 380,
               'template': '...',
               'title': {'text': 'Choice Probabilities by Traveler Type'},
               'xaxis': {'title': {'text': 'Traveler Type'}},
               'yaxis': {'title': {'text': 'Probability'}}}
})

HTML(value='<b>Utility Decomposition (Selected Event)</b>')

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…